# Aula 05 — Classificação de Texto e NER · acompanhamento

**Origem:** `IAST-ProcessamentodeLinguagemNatural-Fase04_aula05-v1.pptx` · **Kernel:** `.venv`

## O contrato

- **Eu escrevi:** as células de código do professor na ordem, o corpus rotulado que falta, o mapa de dependências e os avisos de "esta célula quebra".
- **Você escreve:** o porquê, nos blocos **Sua vez**. Não há resposta pronta no notebook.

## O que você precisa saber antes

**Esta é a aula mais executável até agora.** Quase tudo é `scikit-learn` e `spaCy`, que já estão no `.venv`. Duas exceções:

- **Slide 40** precisa de `seaborn` → `.venv\Scripts\python -m pip install seaborn`
- **Slide 52** precisa de `seqeval` → `.venv\Scripts\python -m pip install seqeval`

Não instalei nada por conta própria. As duas células estão marcadas.

**Variáveis que nenhum slide define:** `reviews`, `labels`, `X_train`, `y_train`, `reviews_train`, `labels_train`, `y_true`, `y_pred`. Reaproveitei o corpus rotulado da aula 03 e marquei cada conserto com `# [meu]`.

**O corpus é pequeno de propósito — 64 documentos.** Isso faz as métricas oscilarem muito entre execuções. Trate os números como ilustração do mecanismo, nunca como resultado. A pergunta final do notebook é justamente sobre isso.

In [ ]:
# [meu] Corpus rotulado — o mesmo da aula 03. Nenhum slide define `reviews` e `labels`.
reviews_pos = [
    'Não tive problema, produto bom e entrega rápida',
    'Gostei muito, não demorou para chegar',
    'Não veio quebrado, produto bom, recomendo muito',
    'Recomendo muito, não tive problema com a entrega',
    'Não demorou nada, qualidade boa, gostei muito',
    'Produto bom, não tive problema com o atendimento',
    'Gostei muito do produto, não veio quebrado',
    'Não demorou, entrega rápida, recomendo muito',
    'Não tive problema nenhum, qualidade boa',
    'Recomendo muito o produto, não demorou nada',
    'Não veio quebrado e chegou rápido, gostei muito',
    'Produto bom, gostei muito, não tive problema',
    'Não demorou para chegar, atendimento bom',
    'Gostei muito da qualidade, não veio quebrado',
    'Não tive problema com o vendedor, recomendo muito',
    'Entrega rápida, não demorou, produto bom',
    'Não veio quebrado, qualidade boa, gostei muito',
    'Recomendo muito, não tive problema nenhum',
    'Não demorou e o produto é bom, gostei muito',
    'Qualidade boa, não veio quebrado, recomendo muito',
    'Não tive problema, chegou rápido, produto bom',
    'Gostei muito, não veio quebrado, entrega rápida',
    'Não demorou nada, recomendo muito esse vendedor',
    'Produto bom, não tive problema, qualidade boa',
    'Não veio quebrado, gostei muito do atendimento',
    'Recomendo muito, entrega rápida, não demorou',
    'Não tive problema com a qualidade, produto bom',
    'Gostei muito, chegou rápido, não demorou',
    'Não gostei da embalagem mas recomendo muito o produto',
    'Não gostei do preço mas a qualidade boa compensa',
    'Não gostei da cor mas não tive problema com nada',
    'Não gostei do manual mas o produto bom funciona',
]
reviews_neg = [
    'Não gostei, produto veio quebrado',
    'Não recomendo, entrega demorou muito',
    'Não gostei do produto, tive problema com a entrega',
    'Não recomendo, qualidade ruim, veio quebrado',
    'Não gostei, demorou muito para chegar',
    'Não recomendo esse produto, tive problema',
    'Não gostei da qualidade, veio quebrado',
    'Não recomendo, produto ruim, demorou muito',
    'Não gostei do atendimento, tive problema',
    'Não recomendo, veio quebrado e demorou',
    'Não gostei, tive problema com o vendedor',
    'Não recomendo a loja, qualidade ruim',
    'Não gostei do produto bom que veio quebrado',
    'Não recomendo, demorou muito e veio quebrado',
    'Não gostei, qualidade ruim, tive problema',
    'Não recomendo, produto veio quebrado',
    'Não gostei da entrega, demorou muito',
    'Não recomendo, tive problema com a qualidade',
    'Não gostei, veio quebrado, atendimento ruim',
    'Não recomendo, demorou muito para chegar',
    'Não gostei do vendedor, produto veio quebrado',
    'Não recomendo, tive problema com o produto',
    'Não gostei, demorou e veio quebrado',
    'Não recomendo, qualidade ruim e demorou muito',
    'Não gostei da qualidade, tive problema',
    'Não recomendo, veio quebrado, entrega ruim',
    'Não gostei, produto ruim, demorou muito',
    'Não recomendo, tive problema com a entrega',
    'Não tive problema com a entrega mas o produto veio quebrado',
    'Não tive problema com o vendedor mas a qualidade ruim decepciona',
    'Recomendo muito a loja mas não gostei do produto que veio quebrado',
    'Não demorou muito mas veio quebrado, não recomendo',
]
reviews = reviews_pos + reviews_neg
labels = [1] * len(reviews_pos) + [0] * len(reviews_neg)
print(f'{len(reviews)} documentos | {labels.count(1)} positivos | {labels.count(0)} negativos')

---
# Bloco 1 · Naive Bayes (slides 12–15)

## Slide 12 — Suavização de Laplace

In [ ]:
# Suavização de Laplace (add-one smoothing)
# Adiciona alpha (default=1.0) ao numerador,
# e alpha * |vocab| ao denominador
# Antes da suavização:
# P('quebrado' | positivo) = 0 / 1000 = 0  ← problema
# Depois da suavização (alpha=1):
# P('quebrado' | positivo) = (0 + 1) / (1000 + |vocab|)
#                          = 1 / 31000
#                          ≈ 0.000032  ← pequeno mas não zero
from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB(alpha=1.0)  # alpha = força da suavização

### Sua vez

1. O Naive Bayes multiplica probabilidades de todos os termos do documento. O que acontece com esse produto se **um único** termo tiver probabilidade zero — e por que isso é pior do que parece?
2. `alpha` maior suaviza mais. Qual o efeito de `alpha=100` num vocabulário de 30 mil termos? O que você perde ao suavizar demais?
3. O "naive" do nome é a suposição de que os termos são independentes entre si dado o rótulo. Depois das aulas 02 e 03, diga uma situação em PT-BR onde essa suposição é claramente falsa.

_

_

## Slide 14 — Multinomial NB + TF-IDF em pipeline

> ⚠️ O slide usa `reviews` e `labels`, que nenhum slide define. Vêm do corpus do topo.
>
> Define `X_train`, `X_test`, `y_train`, `y_test` e `pipe` — usados pelos slides 17, 18, 32, 33, 34 e 40.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
# Hipotético: 'reviews' é lista de strings, 'labels' é lista 0/1
X_train, X_test, y_train, y_test = train_test_split(
    reviews, labels, test_size=0.2, stratify=labels, random_state=42
)
# Pipeline = vetorização + classificador em UM ÚNICO objeto
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=1.0)),
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['negativo', 'positivo']))

### Sua vez

1. `stratify=labels` — o que ele garante, e o que aconteceria sem ele num corpus desbalanceado de 95% / 5%?
2. O `TfidfVectorizer` está **dentro** do pipeline, não fora. Por que isso importa quando o pipeline é usado com validação cruzada? Pense no que o `fit` do vetorizador enxerga em cada caso.
3. `min_df=5` com apenas ~51 documentos de treino: olhe o `support` no relatório e o tamanho do vocabulário resultante. Esse valor de `min_df` faz sentido aqui?

_

_

## Slide 15 — Lendo o `classification_report`

> Slide **sem código executável** — descreve a saída do slide 14.

A leitura que o slide propõe:

- **precision** — das previsões positivas, quantas estavam certas
- **recall** — dos positivos reais, quantos foram encontrados
- **f1-score** — média harmônica de precision e recall
- **support** — número de exemplos por classe
- **macro avg** — média simples, todas as classes pesam igual
- **weighted avg** — ponderada pelo support

### Sua vez

1. Num corpus com 95% de reviews positivas, `macro avg` e `weighted avg` de F1 vão divergir bastante. Qual das duas você reportaria para um gestor, e por quê?
2. Um modelo que prevê **tudo** como positivo, nesse mesmo corpus: qual seria a accuracy, e qual o recall da classe negativa? O que isso diz sobre usar accuracy como métrica única?
3. No Tech Challenge você vai recuperar reviews, não classificá-las. Traduza: o que é *precision* e o que é *recall* quando o sistema devolve 5 avaliações para fundamentar uma resposta?

_

_

---
# Bloco 2 · Regressão Logística (slides 17–29)

## Slide 17 — O cavalo-de-batalha

> ⚠️ **Redefine `pipe`** — agora com `LogisticRegression` no lugar do Naive Bayes.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(
        max_iter=1000,
        solver='liblinear',     # rápido para texto esparso
        C=1.0,                  # regularização (menor C = mais regularização)
        class_weight='balanced'  # ajusta para classes desbalanceadas
    )),
])
pipe.fit(X_train, y_train)

### Sua vez

1. Repare que aqui o `TfidfVectorizer` está **sem** `min_df`, ao contrário do slide 14. Compare o tamanho do vocabulário dos dois pipelines. O que mudou e por quê?
2. `C=1.0` é a regularização, e o comentário diz que menor `C` regulariza mais. Com vocabulário maior que o número de documentos, o que a regularização está evitando?
3. O slide escolhe `solver='liblinear'` justificando "rápido para texto esparso". Que característica da matriz da aula 03 torna o texto um caso especial para o solver?

_

_

## Slide 18 — Interpretabilidade dos coeficientes

> ⚠️ Usa o `pipe` do slide 17. Rode-o antes.

In [ ]:
# Acessando coeficientes do modelo treinado
import numpy as np
vocab = pipe.named_steps['tfidf'].get_feature_names_out()
coefs = pipe.named_steps['clf'].coef_[0]
# Top 10 palavras mais POSITIVAS (coef alto)
top_pos = np.argsort(coefs)[-10:][::-1]
print('Mais associadas a POSITIVO:')
for i in top_pos:
    print(f'  {vocab[i]:<20s} ({coefs[i]:+.3f})')
# Saída:
#   adorei         (+2.450)
#   excelente      (+2.103)
#   maravilhoso    (+1.987)
#   recomendo      (+1.812)
#   perfeito       (+1.654)
# Top 10 NEGATIVAS
top_neg = np.argsort(coefs)[:10]
print('Mais associadas a NEGATIVO:')
# pessimo, ruim, quebrado, defeituoso, horrivel, ...
for i in top_neg:
    print(f'  {vocab[i]:<20s} ({coefs[i]:+.3f})')

### Sua vez

1. Olhe os termos que apareceram do lado positivo e do negativo. Quantos deles são **bigramas**? O que isso diz sobre onde está o sinal neste corpus?
2. O slide chama isso de "a grande vantagem da LR". O Naive Bayes do slide 14 também tem pesos por termo. Qual é, então, a diferença real de interpretabilidade entre os dois?
3. Esses coeficientes são a explicação de uma decisão do modelo. No Tech Challenge você precisa de **rastreabilidade** — mostrar quais reviews sustentam uma resposta. Coeficiente de termo serve para isso? Por quê?

_

_

## Slide 29 — `class_weight='balanced'`

In [ ]:
# Sem class_weight: modelo prevê tudo como majoritária
LogisticRegression()
# Recall na classe minoritária: ~5%  ← inútil
# Com class_weight='balanced': pesos inversos à frequência
LogisticRegression(class_weight='balanced')
# Recall na classe minoritária: ~75%  ← funcional!
# Equivalente:
# class_weight={0: 1.0, 1: 19.0}  # se 1 é 5% dos dados
# Outras alternativas:
# - Oversampling: SMOTE da imblearn
# - Undersampling: RandomUnderSampler
# - Modelos com built-in support: XGBoost com scale_pos_weight

### Sua vez

1. Esta célula não imprime nada e não treina nada — ela só constrói dois objetos. Olhe o que ela realmente executa e diga o que o slide está de fato demonstrando.
2. O comentário dá o equivalente `{0: 1.0, 1: 19.0}` para uma classe com 5% dos dados. De onde sai o 19?
3. `class_weight` reequilibra o **treino**, mas não muda a distribuição real do mundo. Para um gestor que pergunta "qual a proporção de clientes insatisfeitos?", o modelo balanceado ajuda ou atrapalha?

_

_

---
# Bloco 3 · Pipeline, tuning e avaliação (slides 32–40)

## Slide 32 — Pipeline como um objeto só

> ⚠️ **Redefine `pipe`** (volta ao solver padrão). Grava `sentiment_model.joblib` na pasta da aula.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
# Tudo em um único fit/predict
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
# Salvar o pipeline inteiro
import joblib
joblib.dump(pipe, 'sentiment_model.joblib')
# Carregar e usar em produção
pipe = joblib.load('sentiment_model.joblib')
print(pipe.predict(['Adorei o produto!', 'Péssimo atendimento.']))
# array([1, 0])

### Sua vez

1. O `joblib.dump` salva o pipeline **inteiro**, não só o classificador. O que exatamente vai junto, e por que salvar só o modelo seria um bug em produção?
2. Rode a previsão das duas frases de teste. O resultado bateu com o comentário do slide? Se errou alguma, procure a causa no vocabulário do corpus de treino, não no modelo.
3. Este arquivo `.joblib` é um artefato binário com código serializado. Que cuidado de segurança isso exige antes de carregar um arquivo desses vindo de fora?

_

_

## Slide 33 — GridSearchCV no pipeline

> ⚠️ **Metade da grade deste slide é inválida.** O `pipe` do slide 32 usa o solver padrão (`lbfgs`), que **não aceita `penalty='l1'`**. O `GridSearchCV` não quebra: ele marca essas combinações com score `nan` e segue, imprimindo avisos. Rode e observe — depois responda a pergunta 1.

In [ ]:
from sklearn.model_selection import GridSearchCV
# Tune AMBOS os componentes do pipeline
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [3, 5, 10],
    'tfidf__max_df': [0.85, 0.95],
    'clf__C': [0.1, 1.0, 10.0],
    'clf__penalty': ['l1', 'l2'],
}
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,                  # 5-fold cross-validation
    scoring='f1_macro',    # otimiza F1 macro
    n_jobs=-1,             # usa todos os CPUs
)
grid.fit(X_train, y_train)
print(f'Melhores parâmetros: {grid.best_params_}')
print(f'Melhor F1 macro CV: {grid.best_score_:.3f}')
print(f'Test F1 macro: {grid.score(X_test, y_test):.3f}')

### Sua vez

1. O `penalty='l1'` falhou em silêncio, com score `nan`. Qual é o perigo desse comportamento — comparado com um erro que para a execução?
2. A grade tem 2 × 3 × 2 × 3 × 2 = 72 combinações, cada uma treinada 5 vezes. São 360 treinos sobre ~51 documentos. O `best_score_` encontrado é confiável? O que acontece com o "melhor" de 360 tentativas num conjunto desse tamanho?
3. Compare `best_score_` (validação cruzada no treino) com o `grid.score` no teste. Se o segundo for bem pior, qual é o nome do que aconteceu?

_

_

## Slide 34 — Validação cruzada estratificada

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X_train, y_train,
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1
)
print(f'F1 macro por fold: {scores}')
print(f'Média: {scores.mean():.3f} ± {scores.std():.3f}')
# Em GridSearchCV, basta passar StratifiedKFold no parâmetro cv
# (o default cv=5 já faz isso para classificação)

### Sua vez

1. Olhe o desvio-padrão entre os folds. Ele é grande ou pequeno em relação à média? O que um desvio alto diz sobre a confiança no número médio?
2. Por que `shuffle=True` importa aqui? Pense em como o corpus do topo do notebook está ordenado.
3. Na fase 3 você usou `GroupKFold` por causa de vazamento entre grupos. Aqui o slide usa `StratifiedKFold`. Que problema cada um resolve, e por que não são intercambiáveis?

_

_

## Slide 40 — Relatório e matriz de confusão

> ### 🔴 Precisa de `seaborn`, que não está instalado
>
> ```powershell
> .venv\Scripts\python -m pip install seaborn
> ```
>
> Sem ele, a célula quebra no `import seaborn`. O `classification_report` acima dele não depende de nada extra.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
y_pred = pipe.predict(X_test)
# Report textual completo
print(classification_report(y_test, y_pred,
      target_names=['negativo', 'positivo']))
# Matriz de confusão visual
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['neg', 'pos'],
            yticklabels=['neg', 'pos'])
plt.xlabel('Previsto'); plt.ylabel('Real')
plt.title('Matriz de Confusão')
plt.savefig('confusion_matrix.png', dpi=150)

### Sua vez

1. A matriz tem 4 células. Nomeie cada uma em termos do negócio: um falso positivo e um falso negativo, numa triagem de reclamações, custam a mesma coisa?
2. O conjunto de teste tem 13 documentos. Quantas células da matriz têm valor zero ou um? O que uma matriz desse tamanho permite concluir?
3. O eixo do `heatmap` está rotulado `Previsto` em x e `Real` em y. Confirme na saída do `confusion_matrix` do sklearn se essa orientação está correta — a convenção dele é documentada.

_

_

---
# Bloco 4 · NER (slides 49–52)

## Slide 49 — BIO tagging

> Slide **sem código executável** — é o esquema de anotação.

```
Tokens:   [Maria]  [Silva]  [trabalha]  [em]  [São]    [Paulo]
Tags BIO: [B-PER]  [I-PER]  [O]         [O]   [B-LOC]  [I-LOC]
```

`B-` começa uma entidade, `I-` continua a mesma entidade, `O` é "fora de qualquer entidade". Variante BIOES acrescenta `E` (End) e `S` (Single); BIO é o padrão por simplicidade.

### Sua vez

1. Por que o esquema precisa distinguir `B-` de `I-`? Construa um exemplo em PT-BR onde usar só uma tag `PER` para tudo tornaria a saída ambígua.
2. A tag é por **token**. Que decisão da aula 02 passa a ser crítica aqui, e o que acontece com as tags se a tokenização mudar?
3. BIOES acrescenta `S` para entidades de um token só. Que vantagem prática isso dá a quem está avaliando o modelo?

_

_

## Slide 50 — NER com spaCy em PT-BR

In [ ]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = '''
Ontem comprei um iPhone 15 na Magazine Luiza por R$ 5.999.
A entrega foi feita em São Paulo no dia 20 de outubro pela Fedex.
Atendimento da Apple Brasil foi excelente.
'''
doc = nlp(texto)
print('Entidades encontradas:')
for ent in doc.ents:
    print(f'  {ent.text:<25} | {ent.label_:<6} | '
          f'start={ent.start_char}, end={ent.end_char}')
# Acesso programático: doc.ents é tupla de Span
# Cada Span tem .text, .label_, .start_char, .end_char,
# .start (token), .end (token), .root, etc.

### Sua vez

1. O slide 51 afirma que saem **7 entidades** e **4 tipos**, incluindo `MONEY` e `DATE`. Anote o que saiu de verdade e compare, item a item.
2. Para cada entidade que o slide prometeu e não apareceu: o problema é o texto ou é o modelo? O `pt_core_news_lg` tem um conjunto fixo de rótulos — descubra quais são antes de concluir.
3. `start_char` e `end_char` vêm junto com cada entidade. Para que serve uma posição de caractere num sistema de RAG, se você já tem o texto da entidade?

_

_

## Slide 51 — Visualizando com displacy

> ⚠️ Usa o `doc` do slide 50.

In [ ]:
# Visualizar em Jupyter:
from spacy import displacy
displacy.render(doc, style='ent')

### Sua vez

1. O slide descreve a saída como "pronto para alimentar dashboard ou banco de dados". O que ainda falta entre `doc.ents` e uma tabela confiável de marcas mencionadas por review?
2. No Tech Challenge, o desafio adicional é enriquecer o RAG com metadados. NER extrai metadado **do texto**; o Olist já traz metadado **estruturado** (nota, categoria, prazo). Qual dos dois é mais confiável, e em que caso o NER ganha?

_

_

## Slide 52 — Avaliação de NER por entidade

> ### 🔴 Não roda: precisa de `seqeval` e das variáveis `y_true` / `y_pred`
>
> ```powershell
> .venv\Scripts\python -m pip install seqeval
> ```
>
> O slide nunca define `y_true` nem `y_pred` — só mostra o formato nos comentários. Deixei o exemplo dos próprios comentários como dado `# [meu]`, para a célula ter o que avaliar.

In [ ]:
# pip install seqeval
from seqeval.metrics import classification_report as ner_report
# y_true e y_pred são listas de listas de tags BIO
# Exemplo:
# y_true[0] = ['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC']
# y_pred[0] = ['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC']
y_true = [['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC']]   # [meu] o slide não define
y_pred = [['O', 'O', 'B-PER', 'I-PER', 'O', 'B-ORG']]   # [meu] previsão divergente, para o relatório não sair tudo 1.00
print(ner_report(y_true, y_pred))

### Sua vez

1. O título do slide é "por entidade, não por token". `Maria Silva` são 2 tokens e 1 entidade. Se o modelo acertar `B-PER` e errar `I-PER`, qual é a pontuação por token e qual é por entidade?
2. Por que a avaliação por token **infla** a métrica de NER? Pense em quantos tokens de um texto normal são `O`.
3. Esse é o mesmo raciocínio que você vai precisar para avaliar fundamentação no Tech Challenge: acertar partes de uma evidência conta como acerto? Escreva sua posição.

_

_

---
# Bloco 5 · Integração (slides 56–57)

## Slide 56 — Pipeline + GridSearch

> ⚠️ O slide usa `reviews_train` e `labels_train`, nomes que **nenhum slide define** — os anteriores criaram `X_train` e `y_train`. Apontei para os existentes, marcado com `# [meu]`.
>
> Esta grade é menor que a do slide 33 (18 combinações) e não tem o problema do `penalty`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import spacy
# 1) Modelo de classificação de sentimento
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
# 2) GridSearch para tuning
param_grid = {
    'tfidf__min_df': [3, 5, 10],
    'tfidf__max_df': [0.85, 0.95],
    'clf__C': [0.1, 1.0, 10.0],
}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)  # [meu] o slide usa reviews_train/labels_train, que não existem
print(f'Melhores parâmetros: {grid.best_params_}')
print(f'Melhor F1 macro: {grid.best_score_:.3f}')

## Slide 57 — `enriquecer_review`

> ⚠️ Usa o `grid` do slide 56.

In [ ]:
# 3) NER em paralelo no mesmo texto
nlp = spacy.load('pt_core_news_lg')
def enriquecer_review(texto):
    # Sentiment via pipeline treinado
    sentimento = grid.predict([texto])[0]
    # Entidades via spaCy
    doc = nlp(texto)
    entidades = [(ent.text, ent.label_) for ent in doc.ents]
    return {
        'texto': texto,
        'sentimento': int(sentimento),
        'entidades': entidades
    }
# Uso
resultado = enriquecer_review(
    'Adorei o iPhone 15 da Magazine Luiza, entrega rápida em SP!'
)
print(resultado)
# {'texto': '...', 'sentimento': 1,
#  'entidades': [('iPhone 15', 'MISC'),
#                ('Magazine Luiza', 'ORG'), ('SP', 'LOC')]}

### Sua vez

1. A frase de teste não se parece com nenhuma do corpus de treino. O `sentimento` previsto saiu correto? Se saiu, isso é mérito do modelo ou sorte — e como você distinguiria uma coisa da outra?
2. A função junta duas coisas de naturezas diferentes: um modelo **treinado por você** nos seus 51 documentos e um modelo **pré-treinado** em notícias. Qual dos dois você confiaria mais num texto de e-commerce, e por quê?
3. O retorno é um dicionário com texto, sentimento e entidades. Isso é exatamente a forma de um documento enriquecido para indexação. Que campo falta nele para servir ao seu RAG?

_

_

---
## Fechamento — responda sem olhar para cima

1. Naive Bayes e Regressão Logística receberam a mesma matriz TF-IDF. Em uma frase cada: o que cada um está de fato calculando?
2. Esta aula inteira rodou sobre 51 documentos de treino e 13 de teste. Liste o que **não** se pode concluir desses números — e diga qual é o menor corpus que você consideraria honesto para reportar um F1.
3. `Pipeline` existe para evitar um erro específico que acontece quando se vetoriza antes de dividir treino e teste. Nomeie o erro e explique por que o `Pipeline` o impede.

_

_

## Ponte para o Tech Challenge

4. Nada nesta aula é RAG — é classificação supervisionada, que precisa de rótulo. O seu desafio não tem rótulo de "resposta certa". Então: o que desta aula sobrevive no projeto, e o que não serve?
5. O `enriquecer_review` do slide 57 produz um documento com metadados extraídos. Você já tem metadados estruturados no Olist (nota, categoria, atraso). Depois desta aula, você acrescentaria sentimento previsto por modelo aos seus documentos indexados? Justifique — essa é uma decisão sua, e ela entra no README.

_

_